# Qwen3-Coder-30B-A3B-Instruct GGUF — H3 ReVeal/test first 1,137

This notebook uses the pre-quantized GGUF Q4 model with llama.cpp. It avoids Transformers converting the full BF16 checkpoint to bitsandbytes at load time, which exhausted GPU 1 on the two-T4 Kaggle runtime. The model is the Qwen3-Coder-30B-A3B-Instruct family in Unsloth UD-Q4_K_XL quantization (~17.7 GB); quantization may slightly change outputs versus BF16.

Attach the same private prompts dataset as before. Restart the Kaggle session first so the failed Transformers load releases VRAM. Then import this notebook and run cells top to bottom. Enable GPU and Internet.

The model artifact is available at [Unsloth GGUF](https://huggingface.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF); llama.cpp supports splitting layers across GPUs ([multi-GPU guide](https://github.com/ggml-org/llama.cpp/blob/master/docs/multi-gpu.md)).

In [ ]:
from pathlib import Path
import os, json, hashlib, subprocess, time, requests, zipfile
from tqdm.auto import tqdm

MODEL_REPO = "unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF"
MODEL_FILE = "Qwen3-Coder-30B-A3B-Instruct-UD-Q4_K_XL.gguf"
MODEL_ALIAS = "qwen3-coder-30b-q4"
EXPECTED_N = 1137
EXPECTED_CONTEXT_BYTES = 8192
MAX_NEW_TOKENS = 256
SEED = 42
INPUT_BASENAME = "prompts_first1137.jsonl"
OUTPUT_DIR = Path("/kaggle/working/qwen3coder30b_gguf_reveal_first1137")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


## Verify the saved H3 prompts

This file is already the first 1,137 prompts in original order, fitted to the 8,192-byte prompt profile. The source hash can differ from the prepared prompt hash after context refitting; validate the prepared hash only.

In [ ]:
candidates = list(Path('/kaggle/input').rglob(INPUT_BASENAME))
if not candidates:
    raise FileNotFoundError(f'Attach a Kaggle Dataset containing {INPUT_BASENAME}.')
PROMPTS_PATH = candidates[0]
rows = [json.loads(line) for line in PROMPTS_PATH.open(encoding='utf-8') if line.strip()]
assert len(rows) == EXPECTED_N, f'Expected {EXPECTED_N}; found {len(rows)}'
for i, row in enumerate(rows):
    prompt = row.get('prompt')
    assert isinstance(prompt, str) and prompt.strip(), f'Missing prompt at row {i}'
    digest = hashlib.sha256(prompt.encode('utf-8')).hexdigest()
    assert digest == row.get('prompt_sha256'), f'Prepared prompt hash mismatch at row {i}'
    trunc = row.get('prompt_truncation', {})
    assert trunc.get('context_limit') == EXPECTED_CONTEXT_BYTES, f'Context mismatch at row {i}'
    assert len(prompt.encode('utf-8')) <= EXPECTED_CONTEXT_BYTES, f'Prompt too long at row {i}'
assert len({r['record_id'] for r in rows}) == EXPECTED_N, 'Duplicate record IDs'
print('Verified', len(rows), 'prompts from', PROMPTS_PATH)
print('First:', rows[0]['record_id'], '| last:', rows[-1]['record_id'])


## Build llama.cpp with CUDA

This compiles the server once in the Kaggle session. If this cell says `nvcc` is missing, stop and send me that error; the notebook image needs a CUDA-enabled build path.

In [ ]:
LLAMA_SRC = Path('/kaggle/working/llama.cpp')
if not shutil.which('cmake'):
    raise RuntimeError('cmake is not available in this Kaggle image.')
if not shutil.which('nvcc'):
    raise RuntimeError('nvcc is not available; this image cannot build llama.cpp CUDA backend as written.')
if not (LLAMA_SRC / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_SRC)], check=True)
subprocess.run([
    'cmake', '-S', str(LLAMA_SRC), '-B', str(LLAMA_SRC / 'build'),
    '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release'
], check=True)
subprocess.run([
    'cmake', '--build', str(LLAMA_SRC / 'build'), '--config', 'Release',
    '--target', 'llama-server', '-j', '2'
], check=True)
SERVER_BIN = LLAMA_SRC / 'build' / 'bin' / 'llama-server'
assert SERVER_BIN.exists(), f'Build did not produce {SERVER_BIN}'
print('Built:', SERVER_BIN)


## Download pre-quantized model

This downloads only the selected Q4 GGUF file (about 17.7 GB), not the original BF16 checkpoint plus temporary conversion tensors.

In [ ]:
%pip -q install -U huggingface_hub requests tqdm
from huggingface_hub import hf_hub_download

MODEL_PATH = Path(hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir='/kaggle/working/qwen3_gguf_model',
))
print('Model file:', MODEL_PATH)
print('Size GiB:', round(MODEL_PATH.stat().st_size / 2**30, 2))


## Start the local inference server

The server loads the model once and uses both visible GPUs in layer-split mode. It keeps one active request at a time to limit VRAM use.

In [ ]:
SERVER_LOG = OUTPUT_DIR / 'llama_server.log'
server_log_file = SERVER_LOG.open('w', encoding='utf-8')
server_cmd = [
    str(SERVER_BIN), '-m', str(MODEL_PATH),
    '--alias', MODEL_ALIAS,
    '--n-gpu-layers', 'all',
    '--split-mode', 'layer',
    '--ctx-size', '8192',
    '--parallel', '1',
    '--host', '127.0.0.1',
    '--port', '8080',
]
server = subprocess.Popen(server_cmd, stdout=server_log_file, stderr=subprocess.STDOUT)
BASE_URL = 'http://127.0.0.1:8080'
for attempt in range(600):
    if server.poll() is not None:
        raise RuntimeError(f'llama-server exited ({server.returncode}). See {SERVER_LOG}')
    try:
        response = requests.get(BASE_URL + '/health', timeout=2)
        if response.ok:
            print('Server ready at', BASE_URL)
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise TimeoutError(f'Server did not become ready. See {SERVER_LOG}')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


## Inference with progress and resume

Progress advances over 1,137 rows. Raw output is appended after each response; restarting the cell skips rows already recorded with matching prompt hashes.

In [ ]:
RAW_PATH = OUTPUT_DIR / 'raw_responses.jsonl'
PRED_PATH = OUTPUT_DIR / 'predictions.jsonl'

def read_checkpoint(path):
    out = {}
    if path.exists():
        with path.open(encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    out[row['record_id']] = row
    return out

raw_done = read_checkpoint(RAW_PATH)
pred_done = read_checkpoint(PRED_PATH)
prompt_hash_by_id = {r['record_id']: r['prompt_sha256'] for r in rows}
for rid, saved in raw_done.items():
    if rid not in prompt_hash_by_id or saved.get('prompt_sha256') != prompt_hash_by_id[rid]:
        raise ValueError(f'Checkpoint prompt mismatch for {rid}; move old outputs before restarting.')
print('Already saved:', len(raw_done), '| remaining:', EXPECTED_N - len(raw_done))

def parse_binary(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines and lines[-1] in {'0', '1'}:
        return int(lines[-1]), 'valid'
    return None, 'invalid_output'

with RAW_PATH.open('a', encoding='utf-8', buffering=1) as raw_f, \
     PRED_PATH.open('a', encoding='utf-8', buffering=1) as pred_f:
    for row in tqdm(rows, total=len(rows)):
        rid = row['record_id']
        if rid in raw_done:
            continue
        started = time.time()
        payload = {
            'model': MODEL_ALIAS,
            'messages': [{'role': 'user', 'content': row['prompt']}],
            'temperature': 0.0,
            'max_tokens': MAX_NEW_TOKENS,
            'seed': SEED,
        }
        try:
            response = requests.post(BASE_URL + '/v1/chat/completions', json=payload, timeout=900)
            response.raise_for_status()
            body = response.json()
            text = body['choices'][0]['message']['content'] or ''
            label, status = parse_binary(text)
            item = {
                'record_id': rid, 'dataset': row.get('dataset', 'reveal'), 'split': row.get('split', 'test'),
                'prompt_sha256': row['prompt_sha256'], 'model': MODEL_REPO + ':' + MODEL_FILE,
                'quantization': 'UD-Q4_K_XL', 'context_limit_bytes': EXPECTED_CONTEXT_BYTES,
                'raw_text': text, 'prediction': label, 'status': status,
                'elapsed_seconds': round(time.time() - started, 3),
            }
        except Exception as exc:
            item = {
                'record_id': rid, 'dataset': row.get('dataset', 'reveal'), 'split': row.get('split', 'test'),
                'prompt_sha256': row['prompt_sha256'], 'model': MODEL_REPO + ':' + MODEL_FILE,
                'quantization': 'UD-Q4_K_XL', 'context_limit_bytes': EXPECTED_CONTEXT_BYTES,
                'raw_text': '', 'prediction': None, 'status': 'request_error',
                'error': str(exc)[:1500], 'elapsed_seconds': round(time.time() - started, 3),
            }
        raw_f.write(json.dumps(item, ensure_ascii=False) + '\n')
        raw_done[rid] = item
        if item['prediction'] is not None:
            pred_f.write(json.dumps({
                'record_id': rid, 'prediction': item['prediction'],
                'prompt_sha256': row['prompt_sha256'], 'model': item['model'],
            }, ensure_ascii=False) + '\n')

print('Raw records:', len(raw_done))
print('Valid predictions:', sum(r.get('prediction') is not None for r in raw_done.values()))
print('Raw:', RAW_PATH)
print('Predictions:', PRED_PATH)


In [ ]:
zip_path = Path('/kaggle/working/qwen3coder30b_gguf_reveal_first1137_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.glob('*.jsonl'):
        z.write(path, path.name)
print('Download:', zip_path, '| bytes:', zip_path.stat().st_size)
